# Generación y carga de datos sintéticos

Este notebook genera datos sintéticos con `Faker` para las 7 tablas del modelo
(customers, categories, products, orders, order_items, payments, reviews) y los
carga en el dataset de BigQuery creado en `01_setup_bigquery.ipynb`.

Fijamos semillas aleatorias (`seed(42)`) para que la generación sea reproducible:
cualquiera que ejecute este notebook obtiene el mismo dataset.

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from google.cloud import bigquery
from google.oauth2 import service_account
import os
import random
from datetime import timedelta
from dotenv import load_dotenv

load_dotenv()
fake = Faker()
Faker.seed(42)
random.seed(42)
np.random.seed(42)

credentials = service_account.Credentials.from_service_account_file(
    os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
)
client = bigquery.Client(project=os.getenv("GCP_PROJECT_ID"), credentials=credentials)
dataset_id = f"{client.project}.{os.getenv('BQ_DATASET_ID')}"
print("Conectado a:", dataset_id)

Conectado a: tc-sql-maria-muriel.techmuriel


## Generación de los datos

Generamos las tablas en el mismo orden de dependencias que en el setup: primero
las que no dependen de ninguna otra (`categories`, `customers`, `products`), y
después las que sí (`orders`, `order_items`, `payments`, `reviews`).

In [2]:
categories_data = [
    "Smartphones", "Laptops", "Tablets", "Audio", "Wearables",
    "Accesorios", "Periféricos", "Componentes"
]

df_categories = pd.DataFrame({
    "category_id": range(1, len(categories_data) + 1),
    "name": categories_data,
    "description": [f"Productos de la categoría {c}" for c in categories_data]
})
df_categories

,category_id,name,description
0,1,Smartphones,Productos de la categoría Smartphones
1,2,Laptops,Productos de la categoría Laptops
2,3,Tablets,Productos de la categoría Tablets
3,4,Audio,Productos de la categoría Audio
4,5,Wearables,Productos de la categoría Wearables
5,6,Accesorios,Productos de la categoría Accesorios
6,7,Periféricos,Productos de la categoría Periféricos
7,8,Componentes,Productos de la categoría Componentes


In [3]:
countries_europe = ["Spain", "France", "Germany", "Italy", "Portugal", "Netherlands"]
channels = ["organic", "paid_ads", "social_media", "referral", "email"]

n_customers = 500
df_customers = pd.DataFrame({
    "customer_id": range(1, n_customers + 1),
    "first_name": [fake.first_name() for _ in range(n_customers)],
    "last_name": [fake.last_name() for _ in range(n_customers)],
    "email": [fake.unique.email() for _ in range(n_customers)],
    "country": [random.choice(countries_europe) for _ in range(n_customers)],
    "city": [fake.city() for _ in range(n_customers)],
    "acquisition_channel": [random.choice(channels) for _ in range(n_customers)],
    "registration_date": [fake.date_between(start_date="-2y", end_date="-1M") for _ in range(n_customers)]
})
df_customers.head()

,customer_id,first_name,last_name,email,country,city,acquisition_channel,registration_date
0,1,Danielle,Dean,pollardmichael@example.net,Netherlands,Caseystad,organic,2025-10-14
1,2,Angel,Johnson,bruce43@example.org,Spain,Stephensfort,referral,2026-05-29
2,3,Joshua,Long,ndavis@example.net,Spain,Robertahaven,organic,2026-04-14
3,4,Jeffrey,Elliott,williamsmatthew@example.org,Netherlands,Paulburgh,referral,2026-06-23
4,5,Jill,Johnson,caseyhubbard@example.org,Germany,West Carlatown,paid_ads,2024-12-17


In [4]:
n_products = 70
df_products = pd.DataFrame({
    "product_id": range(1, n_products + 1),
    "category_id": [random.choice(df_categories["category_id"]) for _ in range(n_products)],
    "name": [fake.unique.catch_phrase() for _ in range(n_products)],  # nombre de producto genérico
    "cost": np.round(np.random.uniform(10, 800, n_products), 2),
})
# El precio de venta siempre por encima del coste (margen entre 15% y 60%)
df_products["price"] = np.round(df_products["cost"] * np.random.uniform(1.15, 1.6, n_products), 2)
df_products["stock"] = np.random.randint(0, 500, n_products)
df_products["is_active"] = np.random.choice([True, False], n_products, p=[0.9, 0.1])
df_products.head()

,product_id,category_id,name,cost,price,stock,is_active
0,1,8,Team-oriented tangible orchestration,305.89,458.07,51,True
1,2,5,Cross-platform radical project,761.06,943.27,267,True
2,3,3,Innovative 5thgeneration superstructure,588.28,677.98,294,True
3,4,7,Re-engineered homogeneous artificial intelligence,482.94,732.60,385,True
4,5,8,Vision-oriented analyzing throughput,133.25,195.62,386,True


In [5]:
n_orders = 2000
statuses = ["pending", "confirmed", "shipped", "delivered", "cancelled", "returned"]
# distribución realista: la mayoría entregados, pocos cancelados/pendientes
status_weights = [0.05, 0.05, 0.15, 0.65, 0.05, 0.05]

order_dates = [fake.date_between(start_date="-1y", end_date="today") for _ in range(n_orders)]

df_orders = pd.DataFrame({
    "order_id": range(1, n_orders + 1),
    "customer_id": [random.choice(df_customers["customer_id"]) for _ in range(n_orders)],
    "status": random.choices(statuses, weights=status_weights, k=n_orders),
    "shipping_country": [random.choice(countries_europe) for _ in range(n_orders)],
    "shipping_city": [fake.city() for _ in range(n_orders)],
    "order_date": order_dates,
})

# shipped_date y delivered_date solo tienen sentido si el pedido avanzó de estado
def compute_shipped(row):
    if row["status"] in ["shipped", "delivered", "returned"]:
        return row["order_date"] + timedelta(days=random.randint(1, 3))
    return pd.NaT

def compute_delivered(row):
    if row["status"] in ["delivered", "returned"]:
        return row["order_date"] + timedelta(days=random.randint(4, 10))
    return pd.NaT

df_orders["shipped_date"] = df_orders.apply(compute_shipped, axis=1)
df_orders["delivered_date"] = df_orders.apply(compute_delivered, axis=1)
df_orders.head()

,order_id,customer_id,status,shipping_country,shipping_city,order_date,shipped_date,delivered_date
0,1,479,cancelled,Portugal,South Rachaelport,2025-09-11,NaT,NaT
1,2,404,cancelled,Spain,North Fredville,2025-09-18,NaT,NaT
2,3,260,shipped,Germany,South Edward,2025-12-08,2025-12-09,NaT
3,4,137,delivered,Netherlands,New Stephanie,2026-08-27,2026-08-28,2026-09-05
4,5,421,shipped,Spain,West Robertside,2026-03-27,2026-03-30,NaT


In [6]:
order_items = []
item_id = 1

for order_id in df_orders["order_id"]:
    n_items = random.choice([1, 2, 2, 3, 3, 3])  # media ~2.5 productos por pedido
    products_in_order = random.sample(list(df_products["product_id"]), min(n_items, n_products))
    for product_id in products_in_order:
        unit_price = df_products.loc[df_products["product_id"] == product_id, "price"].values[0]
        order_items.append({
            "order_item_id": item_id,
            "order_id": order_id,
            "product_id": product_id,
            "quantity": random.randint(1, 3),
            "unit_price": unit_price,
            "discount": round(random.choice([0, 0, 0, 0.05, 0.10, 0.15]) * unit_price, 2)
        })
        item_id += 1

df_order_items = pd.DataFrame(order_items)
print(f"Total order_items generados: {len(df_order_items)}")
df_order_items.head()

Total order_items generados: 4650


,order_item_id,order_id,product_id,quantity,unit_price,discount
0,1,1,63,1,800.26,0.00
1,2,1,4,3,732.60,36.63
2,3,2,40,2,423.77,0.00
3,4,3,63,1,800.26,80.03
4,5,3,1,2,458.07,0.00


In [7]:
payment_methods = ["card", "paypal", "bank_transfer"]

def payment_status(order_status):
    if order_status == "cancelled":
        return "failed"
    if order_status == "returned":
        return "refunded"
    if order_status == "pending":
        return "pending"
    return "completed"

payments = []
for _, order in df_orders.iterrows():
    items_total = df_order_items[df_order_items["order_id"] == order["order_id"]]
    amount = round((items_total["quantity"] * items_total["unit_price"] - items_total["discount"]).sum(), 2)
    payments.append({
        "payment_id": order["order_id"],  # 1 pago por pedido, reutilizamos el id
        "order_id": order["order_id"],
        "payment_method": random.choice(payment_methods),
        "status": payment_status(order["status"]),
        "amount": amount,
        "payment_date": order["order_date"]
    })

df_payments = pd.DataFrame(payments)
df_payments.head()

,payment_id,order_id,payment_method,status,amount,payment_date
0,1,1,bank_transfer,failed,2961.43,2025-09-11
1,2,2,bank_transfer,failed,847.54,2025-09-18
2,3,3,bank_transfer,completed,1636.37,2025-12-08
3,4,4,bank_transfer,completed,1633.90,2026-08-27
4,5,5,bank_transfer,completed,3691.35,2026-03-27


Aquí el amount se calcula sumando de verdad las líneas de order_items de ese pedido — así el importe del pago cuadra con lo comprado, en vez de ser un número aleatorio suelto.

In [8]:
delivered_order_ids = set(df_orders[df_orders["status"] == "delivered"]["order_id"])
delivered_items = df_order_items[df_order_items["order_id"].isin(delivered_order_ids)]

n_reviews = int(len(delivered_items) * 0.35)
reviewed_items = delivered_items.sample(n=n_reviews, random_state=42)

reviews = []
for i, (_, item) in enumerate(reviewed_items.iterrows(), start=1):
    order_date = df_orders.loc[df_orders["order_id"] == item["order_id"], "order_date"].values[0]
    reviews.append({
        "review_id": i,
        "order_item_id": item["order_item_id"],
        "rating": random.choices([1, 2, 3, 4, 5], weights=[0.03, 0.05, 0.12, 0.35, 0.45])[0],
        "comment": fake.sentence() if random.random() > 0.3 else None,  # comentario opcional
        "review_date": pd.to_datetime(order_date) + timedelta(days=random.randint(5, 20))
    })

df_reviews = pd.DataFrame(reviews)
print(f"Total reviews generadas: {len(df_reviews)} ({len(df_reviews)/len(delivered_items)*100:.1f}% de items entregados)")
df_reviews.head()

Total reviews generadas: 1046 (35.0% de items entregados)


,review_id,order_item_id,rating,comment,review_date
0,1,2166.0,5,Law sit science way threat husband.,2026-06-02
1,2,3032.0,5,NaN,2025-11-04
2,3,1213.0,5,According stock affect different interview.,2026-05-07
3,4,4042.0,4,Talk once should mother president.,2026-02-26
4,5,521.0,4,NaN,2026-02-05


## Cargar los 7 DataFrames a BigQuery

Usamos `WRITE_TRUNCATE` para poder volver a ejecutar esta celda sin acumular
datos duplicados si se regenera el dataset. El orden de carga respeta las
dependencias del modelo (tablas sin FK primero).

In [9]:
def load_dataframe(df, table_name, dataset_id=dataset_id, client=client):
    table_ref = f"{dataset_id}.{table_name}"
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
    job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
    job.result()  # espera a que termine
    print(f"{table_name}: {len(df)} filas cargadas")

In [10]:
load_dataframe(df_categories, "categories")
load_dataframe(df_customers, "customers")
load_dataframe(df_products, "products")
load_dataframe(df_orders, "orders")
load_dataframe(df_order_items, "order_items")
load_dataframe(df_payments, "payments")
load_dataframe(df_reviews, "reviews")

c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


categories: 8 filas cargadas


c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


customers: 500 filas cargadas


c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


...

order_items: 4650 filas cargadas


c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


payments: 2000 filas cargadas


c:\Users\34634\BOOTCAMP IA\tc-sql-maria-muriel\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


reviews: 1046 filas cargadas
